In [1]:
import torch
import torch.nn as nn
import torch.optim as optim 
import torchvision

In [2]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

dataset = datasets.ImageFolder(
    "PetImages",
    transform=transform
)

In [3]:
print(dataset.class_to_idx)

{'Cat': 0, 'Dog': 1}


In [4]:
len(dataset)

24998

In [7]:
from torch.utils.data import random_split , DataLoader

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

trainset, testset = random_split(
    dataset,
    [train_size, test_size]
)

In [8]:
print("Total:", len(dataset))
print("Train:", len(trainset))
print("Test:", len(testset))

Total: 24998
Train: 19998
Test: 5000


In [10]:
train_loader = DataLoader(
    trainset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    testset,
    batch_size=32,
    shuffle=False
)

# Build the CNN 

In [17]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers  = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  #kernel_size = 2 ,stride = 2
    
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
    
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(16*16*128,256),
            nn.ReLU(),
            nn.Linear(256,2)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flattening
        x = self.fc_layers(x)
        return x

In [18]:
model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters())

# Train the model

In [19]:
model.train()
epochs = 5

for epoch in range(epochs):
    epoch_training_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model.forward(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()

    print(f"epoch : {epoch+1} & loss : {epoch_training_loss/len(train_loader)}")

C:\Users\Chetan\anaconda3\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


epoch : 1 & loss : 0.6125267719268799
epoch : 2 & loss : 0.49528711252212526
epoch : 3 & loss : 0.40304974381923675
epoch : 4 & loss : 0.3180407520532608
epoch : 5 & loss : 0.2263446201980114


# Evaluate the model

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

model.eval()

actual_labels = []
predicted_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        actual_labels.extend(labels.numpy())
        predicted_labels.extend(predicted.numpy())


accuracy = accuracy_score(
    actual_labels,
    predicted_labels
)

precision = precision_score(
    actual_labels,
    predicted_labels
)

recall = recall_score(
    actual_labels,
    predicted_labels
)

cm = confusion_matrix(
    actual_labels,
    predicted_labels
)


print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("Confusion Matrix:")
print(cm)

Accuracy : 0.8278
Precision: 0.8534844668345928
Recall   : 0.7988212180746562
Confusion Matrix:
[[2106  349]
 [ 512 2033]]
